<a href="https://colab.research.google.com/github/UnstabLe-Fuzion/GeminiProject/blob/main/Receipt_Scanner_Notebook_(Colab_Jupyter).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 1: Introduction & Overview (Markdown)
# ==============================================================================
"""
# 🧾 Receipt Scanner & Data Extractor Notebook

Welcome! This Google Colab/Jupyter Notebook provides a step-by-step guide to building a lightweight receipt scanning and data extraction tool.

**How to use this notebook:**
1.  Run each cell in order from top to bottom by clicking the "Play" button on the left of each cell.
2.  Read the descriptions to understand what each block of code does.
3.  Observe the output of each cell to see the system working in real-time.
4.  Once you've run everything, you can upload your own files to the `input/new` folder and re-run the final "Workflow" cell to process them.

Let's get started!
"""

# ==============================================================================
# CELL 2: Setup - Install Dependencies
# ==============================================================================
"""
This cell installs all the necessary Python libraries and system packages.

- `pytesseract`: The Python wrapper for Google's Tesseract OCR engine.
- `pdf2image`: A library to convert PDFs into images that Tesseract can read.
- `poppler-utils`: A system dependency required by `pdf2image` to work with PDFs on Linux-based systems like Google Colab.
"""
print("⏳ Installing dependencies...")
!pip install pytesseract pdf2image > /dev/null
!apt-get install poppler-utils > /dev/null
print("✅ Dependencies installed successfully!")


# ==============================================================================
# CELL 3: Setup - Create Directory Structure
# ==============================================================================
import os
import shutil

print("⏳ Creating project directories...")

# Define all the paths we'll need
dirs = [
    "input/new",
    "output/json",
    "output/csv",
    "output/logs",
    "archive/receipts",
    "archive/invoices",
    "archive/unknown",
    "temp_images" # For storing images converted from PDFs
]

# Clean up any old directories and create fresh ones
if os.path.exists("input"):
    shutil.rmtree("input")
if os.path.exists("output"):
    shutil.rmtree("output")
if os.path.exists("archive"):
    shutil.rmtree("archive")
if os.path.exists("temp_images"):
    shutil.rmtree("temp_images")

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Directory structure created:")
!ls -R

# ==============================================================================
# CELL 4: Setup - Create Configuration File
# ==============================================================================
import json

print("⏳ Creating configuration file...")

# In Colab, Tesseract is installed system-wide, so we don't need a specific path.
# We define regex rules here. These are the core of our data extraction logic.
# Feel free to modify these rules to better match your documents.
config_data = {
  "tesseract_cmd_path": None, # Not needed in Colab, but good practice to have
  "input_dir": "input/new",
  "output_dir_json": "output/json",
  "output_dir_csv": "output/csv",
  "archive_dir": "archive",
  "temp_dir": "temp_images",
  "log_file": "output/logs/app.log",
  "supported_image_formats": [".jpg", ".jpeg", ".png", ".tiff"],
  "supported_pdf_formats": [".pdf"],
  "extraction_rules": {
    "vendor": "(?i)(?:Store|From|Vendor|Sold by):\\s*([\\w\\s.&'-]+)",
    "date": "(\\d{1,2}[/-]\\d{1,2}[/-]\\d{2,4})",
    "total": "(?i)(?:Total Due|Amount|Total|Balance):?\\s*\\$?([\\d,]+\\.\\d{2})",
    "tax": "(?i)(?:Tax|GST|VAT):\\s*\\$?([\\d,]+\\.\\d{2})"
  }
}

# Write the configuration to a JSON file
with open("ocr_config.json", "w") as f:
    json.dump(config_data, f, indent=4)

print("✅ ocr_config.json created successfully:")
!cat ocr_config.json


# ==============================================================================
# CELL 5: Setup - Download a Sample Receipt Image
# ==============================================================================
import urllib.request

print("⏳ Downloading sample receipt for testing...")

# This is a publicly available, simple sample receipt image.
url = 'https://raw.githubusercontent.com/gregarious-repo/receipt-images/main/sample-receipt-01.jpeg'
sample_receipt_path = os.path.join(config_data['input_dir'], 'sample_receipt_01.jpeg')

try:
    urllib.request.urlretrieve(url, sample_receipt_path)
    print(f"✅ Sample receipt downloaded to: {sample_receipt_path}")
    # Verify by listing files in the input directory
    !ls -l input/new/
except Exception as e:
    print(f"❌ Failed to download sample receipt: {e}")


# ==============================================================================
# CELL 6: Core Logic - Utility & Logging Functions
# ==============================================================================
import logging
from pathlib import Path

print("Defining utility functions...")

def setup_logging(log_file):
    """Configures the application-wide logger to output to both file and console."""
    # Get the root logger
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)

    # Clear existing handlers
    if logger.hasHandlers():
        logger.handlers.clear()

    # Create formatters
    log_format = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

    # Create file handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(log_format)
    logger.addHandler(file_handler)

    # Create console handler
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(log_format)
    logger.addHandler(stream_handler)

    return logger

def load_config(config_path='ocr_config.json'):
    """Loads the JSON configuration file."""
    with open(config_path, 'r') as f:
        return json.load(f)

def move_file(source_path, destination_dir):
    """Moves a file to a new directory, ensuring the destination exists."""
    dest_path = Path(destination_dir) / Path(source_path).name
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    Path(source_path).rename(dest_path)
    logging.info(f"Archived '{source_path}' to '{dest_path}'")

# --- Let's test them ---
config = load_config()
logger = setup_logging(config['log_file'])
logger.info("✅ Logging and configuration setup complete.")

# ==============================================================================
# CELL 7: Core Logic - Preprocessing Functions
# ==============================================================================
from PIL import Image
import pdf2image

logger.info("Defining preprocessing functions...")

def normalize_image(image_path):
    """
    Opens an image, converts it to grayscale for better OCR accuracy, and returns a PIL Image object.
    """
    try:
        img = Image.open(image_path).convert('L') # 'L' mode is for grayscale
        logger.info(f"Normalized image: {image_path}")
        return img
    except Exception as e:
        logger.error(f"Could not normalize image {image_path}: {e}")
        return None

def pdf_to_images(pdf_path, output_folder):
    """Converts each page of a PDF to a PNG image."""
    try:
        # The poppler_path is not needed in Colab if `poppler-utils` is installed
        images = pdf2image.convert_from_path(pdf_path)
        image_paths = []
        p_pdf = Path(pdf_path)
        for i, img in enumerate(images):
            image_path = Path(output_folder) / f"{p_pdf.stem}_page_{i+1}.png"
            img.save(image_path, 'PNG')
            image_paths.append(str(image_path))
        logger.info(f"Converted PDF '{pdf_path}' to {len(image_paths)} image(s).")
        return image_paths
    except Exception as e:
        logger.error(f"Failed to convert PDF {pdf_path}: {e}")
        return []

logger.info("✅ Preprocessing functions are defined.")


# ==============================================================================
# CELL 8: Inline Test - Preprocessing
# ==============================================================================
from IPython.display import display

logger.info("--- Testing Preprocessing ---")
normalized_img_obj = normalize_image(sample_receipt_path)

if normalized_img_obj:
    logger.info("Image normalized successfully. Displaying grayscale image:")
    display(normalized_img_obj.resize((400, 600))) # Resize for better display in notebook
else:
    logger.error("Failed to normalize the sample image.")

# ==============================================================================
# CELL 9: Core Logic - OCR Engine
# ==============================================================================
import pytesseract

logger.info("Defining OCR engine function...")

def perform_ocr(image_obj, tesseract_cmd_path=None):
    """
    Performs OCR on a PIL Image object using Tesseract.
    """
    try:
        if tesseract_cmd_path:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd_path
        # PSM 6 assumes a single uniform block of text, which works well for many receipts.
        custom_config = r'--oem 3 --psm 6'
        text = pytesseract.image_to_string(image_obj, config=custom_config)
        logger.info("OCR completed successfully.")
        return text
    except Exception as e:
        logger.error(f"Tesseract OCR failed: {e}")
        return ""

logger.info("✅ OCR engine function is defined.")


# ==============================================================================
# CELL 10: Inline Test - OCR
# ==============================================================================
logger.info("--- Testing OCR Engine ---")
ocr_text = ""
if normalized_img_obj:
    ocr_text = perform_ocr(normalized_img_obj)
    if ocr_text:
        logger.info("OCR extracted the following text:")
        print("-------------------- OCR OUTPUT --------------------")
        print(ocr_text)
        print("----------------------------------------------------")
    else:
        logger.error("OCR returned no text.")
else:
    logger.error("Cannot run OCR test, the image object is missing.")


# ==============================================================================
# CELL 11: Core Logic - Document Classifier
# ==============================================================================
logger.info("Defining document classification function...")

def classify_document(ocr_text):
    """Classifies a document based on keywords found in the OCR text."""
    text_lower = ocr_text.lower()
    if "invoice" in text_lower:
        return "invoice"
    # Receipts are more varied, look for common terms
    if "receipt" in text_lower or ("total" in text_lower and "item" in text_lower):
        return "receipt"
    if "statement" in text_lower and "account" in text_lower:
        return "statement"

    logger.warning("Could not definitively classify document, marking as 'unknown'.")
    return "unknown"

logger.info("✅ Classification function is defined.")

# ==============================================================================
# CELL 12: Inline Test - Classification
# ==============================================================================
logger.info("--- Testing Classification ---")
if ocr_text:
    doc_type = classify_document(ocr_text)
    logger.info(f"Document classified as: '{doc_type}'")
else:
    logger.error("Cannot run classification test, OCR text is empty.")


# ==============================================================================
# CELL 13: Core Logic - Field Extractor
# ==============================================================================
import re

logger.info("Defining field extraction function...")

def extract_fields(ocr_text, rules):
    """Extracts fields from OCR text using a dictionary of regex rules."""
    extracted_data = {}
    logger.info("Starting field extraction...")
    for field, pattern in rules.items():
        match = re.search(pattern, ocr_text, re.IGNORECASE | re.MULTILINE)
        if match:
            # Get the first capturing group, which contains the value
            value = match.group(1).strip()
            extracted_data[field] = value
            logger.info(f"  ✅ Extracted '{field}': {value}")
        else:
            extracted_data[field] = None
            logger.warning(f"  ⚠️ Could not find '{field}' in document.")
    return extracted_data

logger.info("✅ Extraction function is defined.")

# ==============================================================================
# CELL 14: Inline Test - Extraction
# ==============================================================================
logger.info("--- Testing Field Extraction ---")
if ocr_text:
    extracted_data = extract_fields(ocr_text, config['extraction_rules'])
    logger.info("Extraction complete. Results:")
    print(json.dumps(extracted_data, indent=2))
else:
    logger.error("Cannot run extraction test, OCR text is empty.")


# ==============================================================================
# CELL 15: The Full Workflow - Putting It All Together
# ==============================================================================
import csv

def run_workflow():
    """
    This function orchestrates the entire document processing pipeline.
    It scans the input directory and processes each file.
    """
    config = load_config()
    logger = setup_logging(config['log_file'])

    input_dir = Path(config['input_dir'])

    logger.info("==================================================")
    logger.info("🚀 STARTING DOCUMENT PROCESSING WORKFLOW 🚀")
    logger.info("==================================================")

    processed_files = 0
    for file_path in list(input_dir.iterdir()): # Use list() to avoid issues with moving files while iterating
        if not file_path.is_file():
            continue

        processed_files += 1
        logger.info(f"--- Processing new file: {file_path.name} ---")

        # Step 1: Preprocess (Handle PDF or Image)
        text_content = ""
        if file_path.suffix.lower() in config['supported_pdf_formats']:
            image_paths = preprocess.pdf_to_images(file_path, config['temp_dir'])
            full_text = []
            for img_path in image_paths:
                img = normalize_image(img_path)
                if img:
                    full_text.append(perform_ocr(img, config.get('tesseract_cmd_path')))
            text_content = "\n".join(full_text)

        elif file_path.suffix.lower() in config['supported_image_formats']:
            img = normalize_image(file_path)
            if img:
                text_content = perform_ocr(img, config.get('tesseract_cmd_path'))

        else:
            logger.warning(f"Unsupported file type: {file_path}. Skipping.")
            continue

        if not text_content.strip():
            logger.error(f"OCR returned no text for {file_path.name}. Archiving as unknown.")
            move_file(file_path, Path(config['archive_dir']) / 'unknown')
            continue

        # Step 2: Classify Document
        doc_type = classify_document(text_content)

        # Step 3: Extract Fields
        data = extract_fields(text_content, config['extraction_rules'])
        data['source_file'] = file_path.name
        data['document_type'] = doc_type

        # Step 4: Save Structured Output (JSON and CSV)
        output_filename = file_path.stem
        json_path = Path(config['output_dir_json']) / f"{output_filename}.json"
        with open(json_path, 'w') as f:
            json.dump(data, f, indent=4)
        logger.info(f"Saved JSON output to {json_path}")

        csv_path = Path(config['output_dir_csv']) / 'master_receipt_data.csv'
        is_new_file = not csv_path.exists()
        with open(csv_path, 'a', newline='') as f:
            fieldnames = ['source_file', 'document_type'] + list(config['extraction_rules'].keys())
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            if is_new_file:
                writer.writeheader()
            writer.writerow(data)
        logger.info(f"Appended data to master CSV at {csv_path}")

        # Step 5: Archive Original File
        archive_subdir = Path(config['archive_dir']) / doc_type
        move_file(file_path, archive_subdir)

    if processed_files == 0:
        logger.warning("No new files found in the input directory to process.")

    logger.info("==================================================")
    logger.info("✅ WORKFLOW FINISHED ✅")
    logger.info("==================================================")

# --- RUN THE WORKFLOW ---
run_workflow()

# ==============================================================================
# CELL 16: Verification - Check the Results
# ==============================================================================
print("--- Verifying final file structure ---")
# The `input/new` directory should be empty, and the file should be in `archive/receipts`
!ls -R

print("\n\n--- Verifying JSON output ---")
# Display the content of the generated JSON file
!cat output/json/sample_receipt_01.json

print("\n\n--- Verifying CSV output ---")
# Display the content of the master CSV file
!cat output/csv/master_receipt_data.csv

print("\n\n--- Verifying Log File ---")
# Display the last 15 lines of the log file
!tail -n 15 output/logs/app.log

⏳ Installing dependencies...
✅ Dependencies installed successfully!
⏳ Creating project directories...
✅ Directory structure created:
.:
archive  input	output	sample_data  temp_images

./archive:
invoices  receipts  unknown

./archive/invoices:

./archive/receipts:

./archive/unknown:

./input:
new

./input/new:

./output:
csv  json  logs

./output/csv:

./output/json:

./output/logs:

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

./temp_images:
⏳ Creating configuration file...
✅ ocr_config.json created successfully:
{
    "tesseract_cmd_path": null,
    "input_dir": "input/new",
    "output_dir_json": "output/json",
    "output_dir_csv": "output/csv",
    "archive_dir": "archive",
    "temp_dir": "temp_images",
    "log_file": "output/logs/app.log",
    "supported_image_formats": [
        ".jpg",
        ".jpeg",
        ".png",
        ".tiff"
    ],
    "supported_pdf_formats": [
      

2025-06-08 14:41:44,390 - INFO - ✅ Logging and configuration setup complete.
2025-06-08 14:41:44,398 - INFO - Defining preprocessing functions...
2025-06-08 14:41:44,400 - INFO - ✅ Preprocessing functions are defined.
2025-06-08 14:41:44,401 - INFO - --- Testing Preprocessing ---
2025-06-08 14:41:44,404 - ERROR - Could not normalize image input/new/sample_receipt_01.jpeg: [Errno 2] No such file or directory: 'input/new/sample_receipt_01.jpeg'
2025-06-08 14:41:44,406 - ERROR - Failed to normalize the sample image.


❌ Failed to download sample receipt: HTTP Error 404: Not Found
Defining utility functions...


2025-06-08 14:41:44,588 - INFO - NumExpr defaulting to 2 threads.
2025-06-08 14:41:44,914 - INFO - Defining OCR engine function...
2025-06-08 14:41:44,916 - INFO - ✅ OCR engine function is defined.
2025-06-08 14:41:44,917 - INFO - --- Testing OCR Engine ---
2025-06-08 14:41:44,919 - ERROR - Cannot run OCR test, the image object is missing.
2025-06-08 14:41:44,921 - INFO - Defining document classification function...
2025-06-08 14:41:44,923 - INFO - ✅ Classification function is defined.
2025-06-08 14:41:44,927 - INFO - --- Testing Classification ---
2025-06-08 14:41:44,928 - ERROR - Cannot run classification test, OCR text is empty.
2025-06-08 14:41:44,931 - INFO - Defining field extraction function...
2025-06-08 14:41:44,933 - INFO - ✅ Extraction function is defined.
2025-06-08 14:41:44,935 - INFO - --- Testing Field Extraction ---
2025-06-08 14:41:44,936 - ERROR - Cannot run extraction test, OCR text is empty.
2025-06-08 14:41:44,939 - INFO - ==========================================

--- Verifying final file structure ---
.:
archive  input	ocr_config.json  output  sample_data  temp_images

./archive:
invoices  receipts  unknown

./archive/invoices:

./archive/receipts:

./archive/unknown:

./input:
new

./input/new:

./output:
csv  json  logs

./output/csv:

./output/json:

./output/logs:
app.log

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

./temp_images:


--- Verifying JSON output ---
cat: output/json/sample_receipt_01.json: No such file or directory


--- Verifying CSV output ---
cat: output/csv/master_receipt_data.csv: No such file or directory


--- Verifying Log File ---
2025-06-08 14:41:44,921 - INFO - Defining document classification function...
2025-06-08 14:41:44,923 - INFO - ✅ Classification function is defined.
2025-06-08 14:41:44,927 - INFO - --- Testing Classification ---
2025-06-08 14:41:44,928 - ERROR - Cannot run classification test, OCR text is empt